# 01 — EDA and static label graph construction

This notebook reads the cached LADI-v2 data, visualises label prevalence/co-occurrence, and saves static train-only label graphs: PMI, conditional-probability, co-occurrence, and identity control.

In [ ]:

from pathlib import Path
import sys, os, json, time

# Locate the project root whether this folder is in /kaggle/working or uploaded as a Kaggle dataset.
candidates = [Path.cwd(), Path('/kaggle/working/ladi_v2_gcn_project')]
if Path('/kaggle/input').exists():
    candidates += list(Path('/kaggle/input').glob('*'))
PROJECT_ROOT = None
for c in candidates:
    if (c / 'src' / 'ladi_config.py').exists():
        PROJECT_ROOT = c
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find project src/. Upload/unzip the full ladi_v2_gcn_project folder first.')
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.ladi_config import LABEL_COLS_V2A
from src.ladi_graphs import save_graphs_from_csv, build_adj_from_labels, graph_stats

CACHE_DIR = Path('/kaggle/working/ladi_v2a_cache')
OUT_DIR = Path('/kaggle/working/ladi_v2_outputs')
GRAPH_DIR = OUT_DIR / 'graphs'
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(CACHE_DIR / 'train_data.csv')
val_df = pd.read_csv(CACHE_DIR / 'val_data.csv')
test_df = pd.read_csv(CACHE_DIR / 'test_data.csv')
label_cols = LABEL_COLS_V2A
print(train_df.shape, val_df.shape, test_df.shape)


In [ ]:
# Label prevalence
prev = train_df[label_cols].mean().sort_values(ascending=True)
plt.figure(figsize=(8, 5))
plt.barh(prev.index, prev.values)
plt.xlabel('Training prevalence')
plt.title('LADI-v2a training label prevalence')
plt.tight_layout()
plt.savefig(GRAPH_DIR / 'label_prevalence.png', dpi=180)
plt.show()

# Positive labels per image
counts = train_df[label_cols].sum(axis=1)
plt.figure(figsize=(7, 4))
plt.hist(counts, bins=range(0, len(label_cols)+2), align='left', rwidth=0.8)
plt.xlabel('Number of positive labels per image')
plt.ylabel('Images')
plt.title('Multi-label density in training split')
plt.tight_layout()
plt.savefig(GRAPH_DIR / 'label_density_hist.png', dpi=180)
plt.show()


In [ ]:
# Co-occurrence heatmap (raw counts)
Y = train_df[label_cols].values.astype(np.float32)
cooc = Y.T @ Y
plt.figure(figsize=(8, 7))
plt.imshow(cooc)
plt.xticks(range(len(label_cols)), label_cols, rotation=90)
plt.yticks(range(len(label_cols)), label_cols)
plt.colorbar(label='co-occurrence count')
plt.title('Training label co-occurrence counts')
plt.tight_layout()
plt.savefig(GRAPH_DIR / 'cooccurrence_heatmap.png', dpi=180)
plt.show()


In [ ]:
# Build and save graphs.
TOPK = 6  # keep strongest neighbours per label; set 0/None for dense graphs.
paths = save_graphs_from_csv(CACHE_DIR / 'train_data.csv', label_cols, GRAPH_DIR, topk=TOPK)
print(paths)

for mode, path in paths.items():
    A = torch.load(path)
    print('\n', mode, A.shape, 'min/max:', float(A.min()), float(A.max()))
    display(graph_stats(A, label_cols))


In [ ]:
# Save a compact prevalence table used later in the report.
summary = pd.DataFrame({
    'label': label_cols,
    'train_prevalence': train_df[label_cols].mean().values,
    'val_prevalence': val_df[label_cols].mean().values,
    'test_prevalence': test_df[label_cols].mean().values,
    'train_support': train_df[label_cols].sum().values.astype(int),
})
summary.to_csv(GRAPH_DIR / 'label_prevalence_summary.csv', index=False)
display(summary.sort_values('train_prevalence', ascending=False))


Next: run **02_baseline_backbone_training.ipynb**. The baseline checkpoint is later reused to initialise the GCN and Dynamic-GCN backbones.